# Fundamentos: representação digital de imagens

Este tutorial é destinado a estudantes que já conhecem arrays NumPy básicos. Ao final, você conseguirá inspecionar imagens grayscale e coloridas, criar arrays sintéticos, reamostrar imagens, comparar interpoladores e quantizar níveis de intensidade.

## Roteiro

1. Instalar e importar a biblioteca.
2. Carregar uma imagem colorida e inspecionar sua representação NumPy.
3. Comparar representação grayscale e BGR.
4. Criar arrays sintéticos.
5. Reduzir e ampliar a resolução com diferentes interpoladores.
6. Quantizar níveis de intensidade.

## 1. Instalação

No Google Colab, execute a célula a seguir antes dos exemplos.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import cv2 as cv
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.image_creator import ImageCreator
from dip_toolkit.modules.image_loader import ImageLoader
from dip_toolkit.modules.image_transformer import ImageTransformer
from dip_toolkit.modules.statistical_tools import StatisticalTools
from dip_toolkit.modules.visualization import Visualization

loader = ImageLoader()
creator = ImageCreator(seed=2026)
transformer = ImageTransformer()
statistics = StatisticalTools()
visualization = Visualization()

## 2. Imagem colorida como array NumPy

A imagem real é obtida somente pelo resolvedor da disciplina. O `ImageLoader` usa OpenCV, portanto a imagem colorida é carregada na ordem **BGR**.

In [ ]:
image_path = download_course_image(
    "astronaut.png",
    output_dir="/content/dip_images",
)
image_bgr = loader.load_image(image_path)

info_bgr = statistics.get_image_info(image_bgr)
info_keys = (
    "height",
    "width",
    "ndim",
    "channels",
    "shape",
    "dtype",
    "nbytes",
    "minimum",
    "maximum",
    "expected_range",
)
for key in info_keys:
    print(f"{key}: {info_bgr[key]}")

In [ ]:
figure, axis = visualization.show_image(
    image_bgr,
    channel_order="bgr",
    title="Imagem colorida: BGR convertido para RGB no Matplotlib",
)

## 3. Grayscale é um array 2D

Ao converter a imagem BGR para grayscale, a representação deixa de ter canais explícitos: o shape passa de `(altura, largura, 3)` para `(altura, largura)`.

In [ ]:
image_gray = cv.cvtColor(image_bgr, cv.COLOR_BGR2GRAY)
print(statistics.get_image_info(image_gray))

figure, axis = visualization.show_image(
    image_gray,
    channel_order="gray",
    title="Imagem em escala de cinza (array 2D)",
)

## 4. Arrays sintéticos

O `ImageCreator` permite criar exemplos controlados. Em `uint8`, uma imagem de uns usa o nível 255, correspondente ao branco.

In [ ]:
zeros = creator.create_zeros_image((80, 120), dtype=np.uint8)
ones = creator.create_ones_image((80, 120), dtype=np.uint8)
constant = creator.create_filled_image((80, 120), value=128, dtype=np.uint8)

figure, axes = visualization.compare_images(
    [zeros, constant, ones],
    ["Zeros", "Valor constante: 128", "Uns: 255"],
    channel_order="gray",
)

## 5. Amostragem e reamostragem

Primeiro reduzimos a resolução; depois ampliamos a mesma imagem com nearest, bilinear e bicúbica. Nearest mantém blocos mais evidentes, enquanto os demais métodos suavizam as transições.

In [ ]:
reduced_bgr = transformer.resample(image_bgr, (128, 128), interpolation="nearest")
nearest_bgr = transformer.resample(reduced_bgr, image_bgr.shape[:2], "nearest")
bilinear_bgr = transformer.resample(reduced_bgr, image_bgr.shape[:2], "bilinear")
bicubic_bgr = transformer.resample(reduced_bgr, image_bgr.shape[:2], "bicubic")

figure, axes = visualization.compare_images(
    [nearest_bgr, bilinear_bgr, bicubic_bgr],
    ["Nearest", "Bilinear", "Bicúbica"],
    channel_order="bgr",
)

## 6. Quantização uniforme

A quantização reduz o conjunto de intensidades possíveis. Para uma imagem `uint8`, quatro níveis produzem valores distribuídos entre 0 e 255.

In [ ]:
quantized_gray = transformer.quantize_uniform(image_gray, levels=4)
print("Níveis após quantização:", np.unique(quantized_gray))

figure, axes = visualization.compare_images(
    [image_gray, quantized_gray],
    ["Grayscale original", "Quantizada em 4 níveis"],
    channel_order="gray",
)

## Exercício e cuidado comum

Experimente quantizar a imagem em 2, 8 e 16 níveis e compare o resultado. Um erro comum é exibir diretamente uma imagem BGR no Matplotlib: as cores ficam trocadas. Sempre informe `channel_order="bgr"` para que a visualização faça a conversão correta.

In [ ]:
# TODO: escolha outro número de níveis e compare com image_gray.
exercise_levels = 8
exercise_quantized = transformer.quantize_uniform(image_gray, levels=exercise_levels)
visualization.compare_images(
    [image_gray, exercise_quantized],
    ["Original", f"Quantizada em {exercise_levels} níveis"],
    channel_order="gray",
)